In [ ]:
import os

import pandas as pd
import redshift_connector

connection = redshift_connector.connect(
    host=os.environ.get("REDSHIFT_HOST", "retail-dw.482716309154.us-west-2.redshift-serverless.amazonaws.com"),
    port=5439, database="mig_redshift_src",
    user=os.environ["REDSHIFT_USER"], password=os.environ["REDSHIFT_PASSWORD"], ssl=True,
)


In [ ]:
query = """-- BI report: RFM segment sizes and value.
SELECT rfm_segment,
       COUNT(*)          AS customers,
       SUM(monetary)     AS total_monetary,
       AVG(frequency)    AS avg_frequency
FROM   mart.rfm_segments
GROUP  BY rfm_segment
ORDER  BY rfm_segment;
"""
rfm_segments = pd.read_sql_query(query, connection)
rfm_segments.head()


In [ ]:
query = """-- BI report: retention share per cohort x month offset.
SELECT cohort_month,
       month_offset,
       cohort_size,
       active_buyers,
       ROUND(100.0 * active_buyers / cohort_size, 2) AS retention_pct
FROM   mart.cohort_retention
ORDER  BY cohort_month, month_offset;
"""
cohort_retention = pd.read_sql_query(query, connection)
cohort_retention.head()


In [ ]:
connection.close()
